# 9.5 Voting Classifiers

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 9: Ensemble Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-09-ensemble-learning/09.05_voting-classifiers.ipynb)

Companion code for Section 9.5 of the book. Run the cells in order from top to bottom.

**Example 9.5.1**

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X_sc = StandardScaler().fit_transform(data.data)
y    = data.target

# Base classifiers (must support predict_proba for soft voting)
clf1 = LogisticRegression(max_iter=1000, random_state=42)
clf2 = RandomForestClassifier(n_estimators=100, random_state=42)
clf3 = GradientBoostingClassifier(n_estimators=100, random_state=42)
clf4 = SVC(probability=True, kernel='rbf', random_state=42)
clf5 = GaussianNB()
clf6 = KNeighborsClassifier(n_neighbors=7)

# ── Individual model performance ──────────────────────────────────
print('Individual Model CV Accuracy:')
individual = {}
for name, clf in [('LogReg',clf1),('RF',clf2),('GradBoost',clf3),
                   ('SVM',clf4),('NaiveBayes',clf5),('KNN',clf6)]:
    sc = cross_val_score(clf, X_sc, y, cv=5, scoring='accuracy')
    individual[name] = sc.mean()
    print(f'  {name:<12}: {sc.mean():.4f} ± {sc.std():.4f}')

# ── Hard Voting ───────────────────────────────────────────────────
hard_voter = VotingClassifier(
    estimators=[('lr',clf1),('rf',clf2),('gb',clf3),('svm',clf4),('nb',clf5),('knn',clf6)],
    voting='hard'
)
hard_sc = cross_val_score(hard_voter, X_sc, y, cv=5, scoring='accuracy')
print(f'\nHard Voting:     {hard_sc.mean():.4f} ± {hard_sc.std():.4f}')

# ── Soft Voting ───────────────────────────────────────────────────
soft_voter = VotingClassifier(
    estimators=[('lr',clf1),('rf',clf2),('gb',clf3),('svm',clf4),('nb',clf5),('knn',clf6)],
    voting='soft'
)
soft_sc = cross_val_score(soft_voter, X_sc, y, cv=5, scoring='accuracy')
print(f'Soft Voting:     {soft_sc.mean():.4f} ± {soft_sc.std():.4f}')

# ── Weighted Soft Voting ──────────────────────────────────────────
# Weight each model by its CV accuracy
weights = list(individual.values())
weights_norm = [w/sum(weights) for w in weights]

weighted_voter = VotingClassifier(
    estimators=[('lr',clf1),('rf',clf2),('gb',clf3),('svm',clf4),('nb',clf5),('knn',clf6)],
    voting='soft',
    weights=weights_norm
)
weighted_sc = cross_val_score(weighted_voter, X_sc, y, cv=5, scoring='accuracy')
print(f'Weighted Soft:   {weighted_sc.mean():.4f} ± {weighted_sc.std():.4f}')

# ── Best 3 models only (drop weakest) ────────────────────────────
top3_voter = VotingClassifier(
    estimators=[('lr',clf1),('gb',clf3),('svm',clf4)],
    voting='soft'
)
top3_sc = cross_val_score(top3_voter, X_sc, y, cv=5, scoring='accuracy')
print(f'Top-3 Soft:      {top3_sc.mean():.4f} ± {top3_sc.std():.4f}')

# ── Visualise comparison ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
all_names   = list(individual.keys()) + ['Hard Vote','Soft Vote','Weighted','Top-3']
all_scores  = list(individual.values()) + [hard_sc.mean(),soft_sc.mean(),weighted_sc.mean(),top3_sc.mean()]
all_colors  = ['#BDBDBD']*6 + ['#1976D2','#43A047','#FF6F00','#9C27B0']

bars = ax.bar(all_names, all_scores, color=all_colors, edgecolor='white')
for bar, sc in zip(bars, all_scores):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
           f'{sc:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.axhline(max(individual.values()), color='red', ls='--', lw=1.5,
          alpha=0.7, label=f'Best individual: {max(individual.values()):.3f}')
ax.set_ylabel('5-Fold CV Accuracy'); ax.set_ylim(0.920, 0.985)
ax.set_title('Voting Classifier Comparison', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3, axis='y')
ax.set_xticklabels(all_names, rotation=30, ha='right')
plt.tight_layout()
plt.savefig('voting_comparison.png', dpi=150, bbox_inches='tight')
plt.show()